In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.gm_selection import build_final_ensembles
from phd_project.scripts.WP1_ground_motion_set.setup_AvgSA03_gm_selection import setup_AvgSA03_gcim_gm_selection


cfg = load_config()

# AvgSA([0, 3])

In [ ]:
# canonical Stage-1 output (consumed by the post-processing notebook)
final_ensembles_fp = cfg["proc_data"]["gm_selection"] / "AvgSA_03_final_ensembles.pickle"

# intermediate stage caches, one per round (provenance-guarded)
GMS = cfg["proc_data"]["gm_selection"]
stage_fps = {
    "select": [
        GMS / "AvgSA_03_prelim_selection.pickle",        # round 1 selection (all sites)
        GMS / "AvgSA_03_prelim_reselection.pickle",      # round 2 reselection (no-ensemble sites)
        GMS / "AvgSA_03_prelim_reselection_rd03.pickle", # round 3 reselection (usually none)
    ],
    "optimise": [
        GMS / "AvgSA_03_optimised_selection_rd01.pickle",  # round 1 optimise
        GMS / "AvgSA_03_optimised_selection_rd02.pickle",  # round 2 optimise
        GMS / "AvgSA_03_optimised_selection_rd03.pickle",  # round 3 optimise (usually skipped)
    ],
}

# inputs used for provenance fingerprinting (hashed by their file bytes)
gcim_dist_fp = cfg["proc_data"]["gcim_dists"] / "gcim_dist_AvgSA_03.pickle"
source_fps = {
    "gm_db_file":        cfg["proc_data"]["gm_database"],
    "gcim_file":         gcim_dist_fp,
    "disagg_data_file":  cfg["proc_data"]["site_hazard"] / "AvgSA_03_disagg_data_60sites.pickle",
    "disagg_stats_file": cfg["proc_data"]["site_hazard"] / "AvgSA_03_disagg_stats_60sites.pickle",
    "site_model_file":   cfg["hazard_models"]["eshm20_AvgSA_site_model_all"],
}

# Selection scheme: 3 rounds, dropping progressively more causal-parameter bounds.
# Shared with AvgSA_06 once that notebook is migrated onto build_final_ensembles.
round_unbounded = [
    [],                  # round 1: all of m, d, vs30 bounded
    ["d"],               # round 2: drop the distance bound
    ["m", "d", "vs30"],  # round 3: drop all bounds (fallback; not needed for AvgSA_03)
]

# Per round: optimise the whole round work-set (True) or only sites still failing
# after that round's selection (False). [False, True, False] reproduces the legacy
# result: round 2 re-optimises the reselected sites; round 3 has nothing to do.
force_optimisation = [False, True, False]

# other stuff:
rng_seed = 1

# Set True to rebuild every stage + the final artifact, ignoring (and overwriting)
# any existing cache. Leave False for normal runs: an input change then raises
# StaleCacheError instead of silently reusing stale results.
FORCE_RECOMPUTE = False

In [4]:
# set up the record selection
site_poe_disaggs, disagg_stats, site_model, basic_selection_ctx, gm_db = setup_AvgSA03_gcim_gm_selection()

# load the gcim distributions
if gcim_dist_fp.is_file():
    with open(gcim_dist_fp, "rb") as file:
        gcim_dists = pickle.load(file)
    print("Existing GCIM distribution data loaded...")
else:
    print("No existing GCIM distribution data found...")

Existing GCIM distribution data loaded...


## Stage 1 - Build final ensembles (slow compute)

Runs the configurable 3-round selection + optimisation engine and saves the canonical
`AvgSA_03_final_ensembles.pickle` (+ a `.manifest.json` provenance sidecar). Each round
drops progressively more causal-parameter bounds (`round_unbounded`), and each step is
provenance-cached: if an input (gm_db, gcim distributions, disagg, site model, params,
rng_seed, pickagm version, round config) changed since the cache was written, a
`StaleCacheError` is raised instead of silently reusing stale data - set
`FORCE_RECOMPUTE = True` to rebuild.

The defaults (`round_unbounded = [[], ["d"], ["m","d","vs30"]]`,
`force_optimisation = (False, True, False)`) reproduce the legacy AvgSA_03 result
exactly; round 3 is a no-op fallback. Progress bars are labelled `R{n}/3 select/optimise`
and cached stages print `[cache] ... loaded`.

Post-processing (result pickles, plots, download/convert CSVs) lives in the separate,
fast notebook **`wp1pt3pt8e-gm_selection_AvgSA_03_stage2_postprocess.ipynb`**.

In [ ]:
final_ensembles = build_final_ensembles(
    site_poe_disaggs,
    disagg_stats,
    gcim_dists,
    gm_db,
    basic_selection_ctx,
    site_model,
    source_fps=source_fps,
    stage_fps=stage_fps,
    output_fp=final_ensembles_fp,
    round_unbounded=round_unbounded,
    force_optimisation=force_optimisation,
    rng_seed=rng_seed,
    force_recompute=FORCE_RECOMPUTE,
)